### Ventures AI

Chat Bot To query data from [Y-Combinator Startup directory](https://www.ycombinator.com/companies)
> Data Source: https://github.com/yc-oss/api (open sourec Y Combinator companies API)

In [1]:
!uv add -r requirements.txt

Resolved 285 packages in 4.21s=1.44.0                                
Checked 255 packages in 41ms


### LLM Evaluation

Read the ingested companies from Postgres (`ventures_db.yc_oss`, loaded by the `yc_oss_to_ventures_db` Kestra flow)
> Get 1/10th of all records for generating ground truths

In [72]:
import pandas as pd
import psycopg

conn = psycopg.connect(
    host="localhost",
    port=5440,
    dbname="ventures_db",
    user="postgres",
    password="postgres",
)

# with conn:
df = pd.read_sql("""
    SELECT 
        company_id id,
        title company_name_desc,
        content
    FROM yc_oss_fulltext
    where random() < 0.1
    """
, conn)

# conn.close()

print(f"Loaded {len(df)} companies")
df.head()

/var/folders/m4/y4ly49q978l8dsj7_7n8ts_40000gp/T/ipykernel_5670/2277306113.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""


Loaded 515 companies


,id,company_name_desc,content
0,402,Mth Sense — We're a mobile on-device profiling...,"mth sense focuses on finding solutions to ""bli..."
1,451,Credictive — Attribution metatags for online c...,"Industries: B2B, Engineering, Product and Desi..."
2,28852,Kite — Screen recorder for stunning product de...,Kite is a screen recorder that makes your demo...
3,27198,Velontra — Hypersonic space plane that can tak...,Velontra is building a hypersonic space plane ...
4,31452,Degla Inc — The fully autonomous intelligence ...,We turn natural Language into Multi-Drone Miss...


In [73]:
df.shape

(515, 3)

##### Save/Load Sample Questions

In [74]:
# df.to_csv('data/samples_company_records.csv')

df = pd.read_csv('data/samples_company_records.csv')

In [75]:
company_records = df.to_dict(orient='records')

In [76]:
company_records[:10]

[{'Unnamed: 0': 0,
  'id': 1754,
  'company_name_desc': 'Mighty Buildings — 3D printing beautiful, high-quality, and sustainable homes.',
  'content': "Mighty Buildings is an innovative construction technology company based in Oakland, CA creating beautiful, sustainable, and high-quality homes using 3d-printing, robotics, and automation. Their mission is to have a positive impact on the environment, local communities, and the housing crisis through their sustainable approach.\r\n\r\nMighty Buildings' technology has the potential to unlock the needed productivity for large scale construction alongside the opportunity for reduced emissions, leading to a more sustainable product and future. Mighty Buildings was founded by a team of physicists and robotics engineers with extensive experience solving hard R&D problems and building successful engineering firms. They started by inventing a new material that is a 3D printing tech that enabled printing of an entire building, not just walls, in 

#### Generating Ground Truth

In [77]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../.env')
openai_client = OpenAI()



In [21]:
prompt_template = """
You emulate an MBA student researching on businesses.
Formulate 5 questions this student might ask based on a company record.
If the content contains the company name, or any identifier to the company, remove the company name/identifier. The questions should NOT contain company identifier (names etc). 
The questions should like a leading question looking at the company description, with a generalized tone
The questions should be complete and neither too short nor too long. If possible, use as fewer words as possible from the record. 

content: {content}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = openai_client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response


In [78]:
from tqdm.auto import tqdm

results = {}

for doc in tqdm(company_records[:3]): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions = generate_questions(doc)
    results[doc_id] = questions

  0%|          | 0/3 [00:00<?, ?it/s]

In [79]:
results

{1754: '[\n    "How does the company\'s innovative use of 3D printing technology address challenges in sustainable construction?",\n    "In what ways has the team’s expertise in physics and robotics contributed to the development of unique building materials?",\n    "What steps are being taken to ensure that the production process maintains a near-zero waste standard?",\n    "How might the company\'s commitment to achieving Net-Zero by 2028 influence its competitive position within the construction industry?",\n    "What implications does the first achievement of certification under the UL 3401 standard have for the company in terms of market credibility?"\n]',
 33056: '["How does the company leverage AI to analyze experimental data for scientific advancements?", "In what ways does the partnership with laboratories enhance the company’s potential for discovering new materials?", "What specific contributions does the company aim to make towards the new Industrial Revolution?", "Consider

#### Save/Load ground truths

In [80]:
# import json

# rows = []
# for (k,v) in results.items():
#     ques = json.loads(v)
#     for q in ques:
#         rows.append({
#             "company_id": k, "question": q
#         })

# if len(rows) > 0:
#     df_generated_questions = pd.DataFrame(rows)

#     df_generated_questions.to_csv('data/ground_truths.csv')

df_generated_questions = pd.read_csv('data/ground_truths.csv')

#### Hit-rates and MRR

In [85]:
from rag_helper import RAGBase
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

sentence_transformer_model = SentenceTransformer('all-MiniLM-L6-v2')

def vec_to_str(vector):
    return '[' + ','.join(str(x) for x in vector) + ']'

class RAGPgVector(RAGBase):

    def __init__(self, embedder, conn, **kwargs):
        super().__init__(index=None, **kwargs)
        self.embedder = embedder
        self.conn = conn

    def text_search(self, query, num_results=5):
        words = word_tokenize(query)
        stop_words = set(stopwords.words('english'))
        filtered_keywords = [w for w in words if w.isalnum() and w.lower() not in stop_words]

        sql = """
            SELECT
                    company_id,
                    title,
                    content,
                    ts_rank(search_vector, query) AS relevance_score
            FROM
                    yc_oss_fulltext,
                    websearch_to_tsquery('english', '{wapi}') AS query
            WHERE search_vector @@ query
            ORDER BY relevance_score desc
            limit {limits}
        """.format(wapi=' or '.join(filtered_keywords), limits=num_results)

        rows = self.conn.execute(sql).fetchall()

        return [
            {'company_id': r[0], 'title': r[1], 'content': r[2], 'relevance_score': r[3]}
            for r in rows
        ]
        

    def vector_search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        query_str = vec_to_str(query_vector)

        rows = self.conn.execute(
            """
            SELECT 
                company_id,
                title,
                content,
                1 - (embedding <=> %s::vector) AS cosine_similarity
            FROM yc_oss_embeddings
            ORDER BY embedding <=> %s::vector
            LIMIT %s
            """,
            (query_str, query_str, num_results)
        ).fetchall()

        return [
            {'company_id': r[0], 'title': r[1], 'content': r[2], 'cosine_similarity': r[3]}
            for r in rows
        ]

[nltk_data] Downloading package stopwords to /Users/Daudi/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/Daudi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/Daudi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##### Relevance functions

In [89]:
def compute_single_relevance(q, search_func):
    doc_id = q["company_id"]
    results = search_func(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["company_id"] == doc_id))

    return relevance

def compute_all_relevance(ground_truth, search_func):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_single_relevance(q, search_func)
        relevance_total.append(relevance)

    return relevance_total

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                score = 1 / (rank + 1)
                total_score = total_score + score
                break

    return total_score / len(relevance)


##### Search

In [87]:

pgIndex = RAGPgVector(
    embedder=sentence_transformer_model,
    conn=psycopg.connect(
        host="localhost",
        port=5440,
        dbname="ventures_db",
        user="postgres",
        password="postgres",
    ),
    llm_client=openai_client,
    prompt_template=prompt_template
)



In [91]:
# pgIndex.text_search('companies building proprietary material')

res = compute_single_relevance({
        'company_id': 1754,
        # 'question': "How does the company's innovative technology significantly alter the traditional construction processes to address sustainability challenges?"
        'question': 'companies building proprietary material'
    }, pgIndex.text_search)
res

[0, 0, 1, 0, 0]

In [68]:
pgIndex.vector_search('companies building proprietary material', 10)

[{'company_id': 30353,
  'title': 'Axal — Service company that designs, sources, and quality-tests custom PCBs',
  'content': 'We help companies get custom PCBs built without having to manage multiple vendors themselves. We review or create PCB designs, find the right manufacturer, coordinate production, inspect and test every board, handle any manufacturing issues, and deliver the finished boards directly to your office.\r\n\r\nOur goal is to reduce manufacturing errors, shorten iteration cycles, and get your boards to you as quickly as possible without compromising on quality.\nIndustries: Industrials, Manufacturing and Ro',
  'cosine_similarity': 0.5200363707564107},
 {'company_id': 25810,
  'title': "Material Depot — India's Fastest Growing Home Decor destination for Floor & Wall Decor",
  'content': 'rapidly growing digital-first ecosystem, Material Depot isn’t just selling materials—we’re building the future of how India designs and lives.\nIndustries: Real Estate and Constructio

In [ ]:
compute_all_relevance